# 05 - WebDAV upload

Use `B2CInstance().webdav` to PUT a small file to the instance's WebDAV `Sites`
share and read it back. The WebDAV client dispatches through the (Basic) auth
strategy's own HTTP layer, which `respx` intercepts -- so no real server or
credentials are used.

Public API: `B2CInstance`, `WebDavClient` (via `instance.webdav`).

In [ ]:
# --- Offline, credential-free setup -------------------------------------------
# Everything below runs with NO real network and NO real credentials. HTTP is
# mocked with respx, all state lives in a throwaway temp dir, and the auth-token
# caches are reset -- mirroring the SDK's own test harness (tests/conftest.py).
import base64
import json
import os
import tempfile
import time
from pathlib import Path

import httpx
import respx

from b2c_tooling_sdk.auth.oauth import reset_oauth_cache_for_testing
from b2c_tooling_sdk.auth.oauth_implicit import reset_implicit_cache_for_testing
from b2c_tooling_sdk.auth.oauth_pkce import reset_pkce_cache_for_testing
from b2c_tooling_sdk.auth.session_store import (
    FileAuthSessionBackend,
    set_auth_session_backend,
)

_tmp = Path(tempfile.mkdtemp(prefix="b2c-nb-"))
(_tmp / "data").mkdir(parents=True, exist_ok=True)
(_tmp / "config").mkdir(parents=True, exist_ok=True)

# Point every config/data dir at the temp dir so we never touch a real user store.
os.environ["XDG_DATA_HOME"] = str(_tmp / "data")
os.environ["XDG_CONFIG_HOME"] = str(_tmp / "config")
os.environ["LOCALAPPDATA"] = str(_tmp / "data")
os.environ.pop("B2C_CONFIG_DIR", None)

# Reset the module-level OAuth token caches for deterministic runs.
reset_oauth_cache_for_testing()
reset_pkce_cache_for_testing()
reset_implicit_cache_for_testing()

# Install a temp-dir file-backed auth-session store as the default.
set_auth_session_backend(FileAuthSessionBackend(_tmp / "store"))
print("Isolated temp dir:", _tmp)

## Build the instance and grab the WebDAV client

WebDAV uses Basic auth (username / access key). `instance.webdav` is a lazily
constructed, cached `WebDavClient`.

In [ ]:
from b2c_tooling_sdk import AuthConfig, B2CInstance, InstanceConfig
from b2c_tooling_sdk.auth import BasicAuthConfig

HOSTNAME = "example.demandware.net"
instance = B2CInstance(
    InstanceConfig(hostname=HOSTNAME),
    AuthConfig(basic=BasicAuthConfig(username="webdav-user", password="access-key")),
)

webdav = instance.webdav
WEBDAV_BASE = f"https://{HOSTNAME}/on/demandware.servlet/webdav/Sites"
target = "Impex/src/instance/hello.txt"
print("Upload URL:", webdav.build_url(target))

## PUT a file, then GET it back

The in-memory WebDAV mock stores the PUT body and serves it back on GET.

In [ ]:
stored: dict[str, bytes] = {}
payload = b"hello from the python sdk"

def _put(request: httpx.Request) -> httpx.Response:
    stored[str(request.url)] = request.content
    return httpx.Response(201)

def _get(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, content=stored.get(str(request.url), b""))

url = webdav.build_url(target)
with respx.mock(assert_all_called=False) as router:
    router.put(url).mock(side_effect=_put)
    router.get(url).mock(side_effect=_get)

    await webdav.put(target, payload, content_type="text/plain")
    round_tripped = await webdav.get(target)

print("Uploaded bytes :", payload)
print("Downloaded bytes:", round_tripped)
assert round_tripped == payload

## Recap

- `instance.webdav.put(...)` uploaded a file over WebDAV.
- `instance.webdav.get(...)` read the same bytes back.
- The transport was mocked, so this ran offline with fake credentials.